In [1]:
import os
import sys

from dotenv import load_dotenv

load_dotenv()

sys.path.append("..")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Only Embeddings

In [2]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

from app.schemas import AgentState

In [3]:
embedder = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
intent_examples = {
    "support": [
        "What is your return policy?",
        "Do you offer a warranty on this product?",
        "How do I get a refund?",
        "What are your store hours?",
    ],
    "fulfillment": [
        "Is this item in stock?",
        "I want to place an order",
        "Where is my order?",
        "Can I cancel my order?",
    ],
    "vision": [
        "Check the shelf status in aisle 3",
        "Is there a stock gap on this shelf?",
        "Analyze this shelf image",
        "How full is the inventory display?",
    ],
}

In [5]:
category_embeddings = {
    category: [embedder.embed_query(ex) for ex in examples]
    for category, examples in intent_examples.items()
}

In [6]:
def classify_intent(query: str):
    query_vec = np.array(embedder.embed_query(query))
    scores = {}
    for category, vecs in category_embeddings.items():
        sims = [
            np.dot(query_vec, np.array(v))
            / (np.linalg.norm(query_vec) * np.linalg.norm(v))
            for v in vecs
        ]
        scores[category] = max(sims)
    best = max(scores, key=scores.get)
    return best, scores[best], scores

In [7]:
test_queries = [
    "Do you have this in blue?",
    "I'd like to return my order from last week",
    "Can you check if shelf 5 needs restocking?",
    "Do you have Cadbury Chocolate?",
    "What are the number of breads present in Shelf",
    "Where is my order?",
    "Delivery partner not responding",
]
for q in test_queries:
    category, confidence, all_scores = classify_intent(q)
    print(f"{q!r} -> {category} ({confidence:.3f})")
    print(f"   all scores: { {k: round(v, 3) for k, v in all_scores.items()} }\n")

'Do you have this in blue?' -> fulfillment (0.435)
   all scores: {'support': np.float64(0.258), 'fulfillment': np.float64(0.435), 'vision': np.float64(0.259)}

"I'd like to return my order from last week" -> fulfillment (0.583)
   all scores: {'support': np.float64(0.476), 'fulfillment': np.float64(0.583), 'vision': np.float64(0.277)}

'Can you check if shelf 5 needs restocking?' -> vision (0.593)
   all scores: {'support': np.float64(0.399), 'fulfillment': np.float64(0.367), 'vision': np.float64(0.593)}

'Do you have Cadbury Chocolate?' -> support (0.263)
   all scores: {'support': np.float64(0.263), 'fulfillment': np.float64(0.253), 'vision': np.float64(0.257)}

'What are the number of breads present in Shelf' -> vision (0.408)
   all scores: {'support': np.float64(0.21), 'fulfillment': np.float64(0.135), 'vision': np.float64(0.408)}

'Where is my order?' -> fulfillment (1.000)
   all scores: {'support': np.float64(0.327), 'fulfillment': np.float64(1.0), 'vision': np.float64(0.344)}

### LLM Based Approach

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

In [9]:
llm = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b")

In [10]:
class IntentOutput(BaseModel):
    category: Literal["support", "fulfillment", "vision"] = Field(
        description="Which agent should handle this query"
    )
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Sentiment of the query"
    )
    reasoning: str = Field(description="Brief reason for the classification")

In [11]:
structured_llm = llm.with_structured_output(IntentOutput)

In [12]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Classify the customer query into one category:
- support: questions about policies, returns, warranties, general help
- fulfillment: questions about orders, stock availability, placing/tracking orders
- vision: questions about shelf status, inventory display, physical store monitoring
Also determine the sentiment.""",
        ),
        ("human", "{query}"),
    ]
)

In [13]:
chain = prompt | structured_llm

In [14]:
for q in test_queries:
    result = chain.invoke({"query": q})
    print(f"{q!r} -> {result}")

'Do you have this in blue?' -> category='fulfillment' sentiment='neutral' reasoning='The user is asking about product availability in a specific color, which relates to stock and ordering.'
"I'd like to return my order from last week" -> category='support' sentiment='neutral' reasoning='The user is requesting to return an order, which relates to return policies and general help.'
'Can you check if shelf 5 needs restocking?' -> category='vision' sentiment='neutral' reasoning='The query asks about the status of a specific shelf and whether it needs restocking, which pertains to shelf status and physical store monitoring.'
'Do you have Cadbury Chocolate?' -> category='fulfillment' sentiment='neutral' reasoning='The user is asking about product availability, which relates to stock and ordering.'
'What are the number of breads present in Shelf' -> category='vision' sentiment='neutral' reasoning='User asks about the count of breads on a shelf, which relates to shelf status/inventory display.

In [15]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

LABEL_MAP = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}


def get_sentiment(query: str) -> str:
    result = sentiment_analyzer(query)[0]
    label = result["label"]
    return LABEL_MAP.get(label, label.lower())


for q in test_queries:
    print(f"{q!r} -> {get_sentiment(q)}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

'Do you have this in blue?' -> neutral
"I'd like to return my order from last week" -> neutral
'Can you check if shelf 5 needs restocking?' -> neutral
'Do you have Cadbury Chocolate?' -> neutral
'What are the number of breads present in Shelf' -> neutral
'Where is my order?' -> neutral
'Delivery partner not responding' -> negative


In [16]:
CONFIDENCE_THRESHOLD = 0.65


def get_intent(query: str) -> dict:
    category, confidence, all_scores = classify_intent(query)
    local_sentiment = get_sentiment(query)

    if confidence >= CONFIDENCE_THRESHOLD:
        return {
            "query": query,
            "category": category,
            "confidence": round(float(confidence), 3),
            "sentiment": local_sentiment,
            "method": "embedding",
        }
    else:
        result = chain.invoke({"query": query})
        return {
            "query": query,
            "category": result.category,
            "confidence": None,
            "sentiment": result.sentiment,
            "method": "llm_fallback",
            "reasoning": result.reasoning,
        }

In [17]:
import pandas as pd

results = [get_intent(q) for q in test_queries]
df = pd.DataFrame(results)
df

,query,category,confidence,sentiment,method,reasoning
0,Do you have this in blue?,fulfillment,NaN,neutral,llm_fallback,The user asks about product color availability...
1,I'd like to return my order from last week,support,NaN,neutral,llm_fallback,"User wants to return an order, which pertains ..."
2,Can you check if shelf 5 needs restocking?,vision,NaN,neutral,llm_fallback,The user asks about the status of a specific s...
3,Do you have Cadbury Chocolate?,fulfillment,NaN,neutral,llm_fallback,"The user is asking about product availability,..."
4,What are the number of breads present in Shelf,vision,NaN,neutral,llm_fallback,The user asks about the quantity of a specific...
5,Where is my order?,fulfillment,1.0,neutral,embedding,NaN
6,Delivery partner not responding,fulfillment,NaN,negative,llm_fallback,The user reports that the delivery partner is ...
